# `cell_text_embedder` (CellTextEmbedderModule)
- **Directory**: `modules/embedding/cell_text_embedder.py`
- **Category**: Logic (Embedding)
- **Role**: 엑셀 직렬화 셀 텍스트 목록을 고속 배치 임베딩하고 아티팩트 저장소에 저장합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().parent.name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.embedding.cell_text_embedder import CellTextEmbedderModule, CellTextEmbedderInputDTO, CellTextEmbedderConfigDTO

mock_encoder = MagicMock()
mock_encoder.model_name = "text-embedding-3-large"
mock_encoder.dimension = 3072
mock_encoder.embed_documents.return_value = [[0.011] * 3072, [0.022] * 3072]
mock_encoder.encode.return_value = [[0.011] * 3072, [0.022] * 3072]

artifact_store = MagicMock()
artifact_store.is_valid.return_value = False
artifact_store.put_streaming.return_value = None

module = CellTextEmbedderModule(encoder=mock_encoder, artifact_store=artifact_store)

sample_input = {
    "file_name": "samsung_2023.xlsx",
    "workbook_hash": "hash_samsung_2023",
    "company_name": "삼성전자",
    "items": [
        {
            "cell_id": "삼성전자:IS:C5",
            "sheet_name": "손익계산서",
            "cell_coord": "C5",
            "row_header": ["영업이익"],
            "column_header": ["2022"],
            "cell_value": "433766",
            "variant": "header_with_value",
            "text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2022 | Cell Value: 433766"
        },
        {
            "cell_id": "삼성전자:IS:D5",
            "sheet_name": "손익계산서",
            "cell_coord": "D5",
            "row_header": ["영업이익"],
            "column_header": ["2023"],
            "cell_value": "65670",
            "variant": "header_with_value",
            "text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: 65670"
        }
    ]
}
input_dto = CellTextEmbedderInputDTO(**sample_input)
output = module.run(input_dto, config=CellTextEmbedderConfigDTO(batch_size=512))
print_io("cell_text_embedder (CellTextEmbedderModule)", sample_input, output)
